# Causal Tracing Analysis

This notebook demonstrates how to use causal tracing to identify which model components are causally responsible for specific behaviors. Causal tracing works by intervening on internal activations and measuring the effect on model outputs.

## Overview
- Setup the causal tracer with a model
- Run causal trace experiments
- Visualize activation paths
- Interpret causal relationships in the model

In [ ]:
# Setup and imports
import torch
from world_model_lens.patching.causal_tracer import CausalTracer
from world_model_lens import HookedWorldModel
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
torch.manual_seed(42)

print("Causal Tracing environment setup complete")

## Initialize the Causal Tracer

First, we need to load a world model and initialize the causal tracer with it. The tracer will allow us to intervene on specific layers and measure the causal effect.

In [ ]:
# Load a world model (example with DreamerV3-style architecture)
model = HookedWorldModel.from_pretrained("dreamerv3 Atari")

# Initialize the causal tracer
tracer = CausalTracer(model)

# Configure tracer settings
tracer.configure(
    layers_to_trace=["encoder", "transition", "decoder", "reward"],
    intervention_type="mean_ablation",
    num_samples=100
)

print(f"Tracer initialized with {len(tracer.layers_to_trace)} layers")
print(f"Layers: {tracer.layers_to_trace}")

## Run Causal Trace

Now we'll run a causal trace on a specific behavior. We'll trace how different layers contribute to predicting the value of a state.

In [ ]:
# Define a trace experiment
from world_model_lens.patching.causal_tracer import TraceConfig

config = TraceConfig(
    prompt="agent sees a goal state",
    target_behavior="predicts_high_value",
    metric="KL_divergence"
)

# Run the causal trace
trace_results = tracer.trace(config)

print("Causal trace completed!")
print(f"Number of layer effects measured: {len(trace_results.layer_effects)}")

## Visualize Activation Paths

Visualize how causal influence flows through the model layers during the traced behavior.

In [ ]:
# Extract layer effects for visualization
layers = list(trace_results.layer_effects.keys())
effects = [trace_results.layer_effects[layer] for layer in layers]

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Bar chart of causal effects by layer
ax1 = axes[0]
colors = ["#2ecc71" if e > 0 else "#e74c3c" for e in effects]
ax1.bar(layers, effects, color=colors)
ax1.set_xlabel("Model Layer")
ax1.set_ylabel("Causal Effect (KL Divergence)")
ax1.set_title("Causal Effect by Layer")
ax1.tick_params(axis="x", rotation=45)

# Plot 2: Heatmap of attention patterns
ax2 = axes[1]
sns.heatmap(trace_results.attention_patterns, ax=ax2, cmap="viridis", annot=False)
ax2.set_title("Attention Patterns During Trace")
ax2.set_xlabel("Destination Position")
ax2.set_ylabel("Source Position")

plt.tight_layout()
plt.savefig("causal_trace_results.png", dpi=150)
plt.show()

print("Visualization saved to causal_trace_results.png")

## Interpret Results

Analyze which components are causally responsible for the traced behavior and extract meaningful interpretations.

In [ ]:
# Identify the most influential layers
top_layers = tracer.get_top_influential_layers(trace_results, k=3)

print("Top 3 Causally Influential Layers:")
for i, (layer, effect) in enumerate(top_layers, 1):
    print(f"  {i}. {layer}: effect = {effect:.4f}")

# Generate interpretation report
interpretation = tracer.interpret(trace_results)
print("\n" + "="*60)
print("INTERPRETATION REPORT")
print("="*60)
print(interpretation)

In [ ]:
# Additional analysis: Compare clean vs corrupted traces
clean_results = tracer.trace(config, clean_run=True)
corrupted_results = tracer.trace(config, clean_run=False)

# Compute the difference
diff_effects = {}
for layer in trace_results.layer_effects:
    clean_effect = clean_results.layer_effects.get(layer, 0)
    corrupted_effect = corrupted_results.layer_effects.get(layer, 0)
    diff_effects[layer] = abs(corrupted_effect - clean_effect)

print("Layer-wise Difference (Clean vs Corrupted):")
for layer, diff in sorted(diff_effects.items(), key=lambda x: -x[1]):
    print(f"  {layer}: {diff:.4f}")

## Save Results

Export the causal trace results for later analysis.

In [ ]:
import json

# Save trace results
trace_results.save("causal_trace_results.json")

# Also save as a summary
summary = {
    "config": {
        "prompt": config.prompt,
        "target_behavior": config.target_behavior,
        "layers_traced": layers
    },
    "top_influential_layers": top_layers,
    "interpretation": interpretation
}

with open("causal_trace_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Results saved successfully!")